# Database Ingestion: Snapshot Rows with Provenance

| Field | Value |
|---|---|
| Stage | Data foundation |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Database ingestion needs an explicit query, row grain, key, and snapshot boundary. The connection itself is not a document contract.

## 30-Second Summary

This notebook opens the repository SQLite fixture in read-only mode, inspects schema, and compares raw table dumps with a bounded joined snapshot of projects and their leads.

## Why This Matters

Unbounded `SELECT *`, unstable row IDs, and many-to-many joins can create stale or duplicated retrieval documents. Query provenance makes the snapshot reproducible and auditable.

## Scope

| Covers | Does not cover |
|---|---|
| Read-only SQLite, schema inspection, bounded join, row IDs, join checks | Live text-to-SQL, CDC, credentials, production database load |


## Mental Model

```text
read-only DB -> inspect schema -> bounded query -> validate grain/join -> row documents + query provenance
```


In [1]:
from pathlib import Path
import sqlite3

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file(): return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")

REPO_ROOT = find_repo_root()
DB_PATH = REPO_ROOT / "05-DataIngestParsing/data/databases/company.db"
connection = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
connection.row_factory = sqlite3.Row
tables = [row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
tables


['employees', 'projects']

## How It Works

We inspect known tables, run a fixed query with explicit columns and ordering, verify the output grain, then serialize each project row with its primary key and lead details. In production, record query/version and snapshot time or change watermark.


## Baseline

The baseline dumps every row from both tables separately. It preserves rows but forces downstream code to infer the relationship and query provenance.


In [2]:
baseline_rows = {
    table: [dict(row) for row in connection.execute(f"SELECT * FROM {table}")]
    for table in tables
}
{table: len(rows) for table, rows in baseline_rows.items()}


{'employees': 4, 'projects': 4}

## Technique Implementation

The technique uses an explicit one-project-per-row join. Qualified aliases prevent column-name collisions, and the project primary key becomes the document ID.


In [3]:
SNAPSHOT_QUERY = """
SELECT
    p.id AS project_id,
    p.name AS project_name,
    p.status AS project_status,
    p.budget AS project_budget,
    e.id AS lead_id,
    e.name AS lead_name,
    e.role AS lead_role,
    e.department AS lead_department
FROM projects AS p
LEFT JOIN employees AS e ON e.id = p.lead_id
ORDER BY p.id
"""
snapshot_rows = [dict(row) for row in connection.execute(SNAPSHOT_QUERY)]
source = DB_PATH.relative_to(REPO_ROOT).as_posix()
project_documents = [
    {
        "id": f"project:{row['project_id']}", "source": source,
        "metadata": row,
        "content": (
            f"Project: {row['project_name']} | Status: {row['project_status']} | "
            f"Budget: {row['project_budget']} | Lead: {row['lead_name']} ({row['lead_role']})"
        ),
    }
    for row in snapshot_rows
]
[(doc["id"], doc["content"]) for doc in project_documents]


[('project:1',
  'Project: RAG Implementation | Status: Active | Budget: 150000.0 | Lead: John Doe (Senior Developer)'),
 ('project:2',
  'Project: Data Pipeline | Status: Completed | Budget: 80000.0 | Lead: Jane Smith (Data Scientist)'),
 ('project:3',
  'Project: Customer Portal | Status: Planning | Budget: 200000.0 | Lead: Mike Johnson (Product Manager)'),
 ('project:4',
  'Project: ML Platform | Status: Active | Budget: 250000.0 | Lead: Jane Smith (Data Scientist)')]

## Controlled Experiment

We verify that the left join neither drops nor multiplies projects, every document has a stable primary-key ID, and the active RAG Implementation project resolves to its expected lead.


In [4]:
project_count = connection.execute("SELECT COUNT(*) FROM projects").fetchone()[0]
active_project = next(doc for doc in project_documents if doc["metadata"]["project_status"] == "Active")
experiment_result = {
    "source_projects": project_count,
    "snapshot_rows": len(snapshot_rows),
    "unique_project_ids": len({row["project_id"] for row in snapshot_rows}),
    "active_project": active_project["metadata"]["project_name"],
    "active_lead": active_project["metadata"]["lead_name"],
}
experiment_result


{'source_projects': 4,
 'snapshot_rows': 4,
 'unique_project_ids': 4,
 'active_project': 'RAG Implementation',
 'active_lead': 'John Doe'}

## Evaluation

The query produces exactly **4 rows for 4 projects** with 4 unique project IDs. The active `RAG Implementation` project resolves to `John Doe`. This validates the fixture join; it does not establish a production refresh or authorization policy.


In [5]:
assert experiment_result == {
    "source_projects": 4, "snapshot_rows": 4, "unique_project_ids": 4,
    "active_project": "RAG Implementation", "active_lead": "John Doe",
}
assert len({doc["id"] for doc in project_documents}) == len(project_documents)
assert connection.execute("PRAGMA query_only").fetchone()[0] == 0  # URI mode enforces read-only at the file layer.
connection.close()
print("Database checks passed for the bounded read-only snapshot.")


Database checks passed for the bounded read-only snapshot.


## Decision Guide

| Need | Pattern |
|---|---|
| Stable knowledge snapshot | Versioned bounded extract |
| Frequently changing facts | Live authorized SQL/tool call |
| Semantic search over text columns | Snapshot rows plus typed metadata |
| Incremental refresh | CDC/watermark with idempotent upsert/delete |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Duplicate documents | Many-to-many join multiplication | Assert grain and unique keys |
| Deleted rows remain searchable | Append-only refresh | Propagate tombstones/deletes |
| Stale answers | Snapshot age hidden | Record freshness/watermark |
| Sensitive rows leak | Authorization after retrieval | Apply row/tenant policy before candidate selection |


## Production Notes

### Observability
Record query/version, row counts, unmatched joins, duplicate keys, watermark, duration, and source snapshot ID.

### Safety and Guardrails
Use least-privilege read credentials, parameterize filters, bound results, and enforce tenant policy at the source.

### Latency and Cost
Prefer incremental extracts for retrieval indexes; reserve live queries for facts whose freshness justifies database load.


## Practice

Insert a fixture project with a missing lead into a copy of the database. Verify left-join behavior and define whether the document is accepted or quarantined.

## Recall

Toggle - Recall: What must a database snapshot record?
Query/version, grain, primary key, source, and freshness boundary.

Toggle - Recall: Why check row counts after a join?
A join can silently drop or multiply the intended population.

## Sources

- [Python `sqlite3` documentation](https://docs.python.org/3/library/sqlite3.html)
- [SQLite URI filenames](https://www.sqlite.org/uri.html)
- Repository fixture: `company.db`

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the bounded SQLite fixture | Add unmatched and many-to-many join fixtures |
